# 02 — Linear algebra

**Workload:** Linear solve, least-squares polynomial fitting, eigendecomposition/PCA, SVD, and QR.

This notebook is executed against the RNP engine. Every output below is
stored in the notebook and visible when rendered on GitHub.

In [1]:
from pathlib import Path
import importlib.util
import sys

PROJECT_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "shim" / "rnp_numpy").is_dir()
)
for path in (PROJECT_ROOT / "harness" / "_redirect", PROJECT_ROOT / "shim"):
    sys.path.insert(0, str(path))

# IPython may preload the oracle NumPy, so clear that namespace before
# executing the exact redirect hook used by examples/run_all.py.
for module_name in list(sys.modules):
    if module_name == "numpy" or module_name.startswith("numpy."):
        del sys.modules[module_name]
redirect_path = PROJECT_ROOT / "harness" / "_redirect" / "sitecustomize.py"
redirect_spec = importlib.util.spec_from_file_location("_rnp_notebook_redirect", redirect_path)
redirect = importlib.util.module_from_spec(redirect_spec)
redirect_spec.loader.exec_module(redirect)
import numpy as np

probe = np.array(0)
print("numpy version:", np.__version__)
print(f"RNP engine active: {np.__name__} ({type(probe).__module__}.{type(probe).__name__})")
assert np.__name__ == "rnp_numpy"

numpy version: 2.5.2
RNP engine active: rnp_numpy (_rnp.ndarray)


## Solve a linear system

Construct a system from known coefficients and recover them with `linalg.solve`.

In [2]:
expected = np.array([1.0, -2.0, 3.0])
system = np.array([[4.0, 1.0, 2.0], [0.0, 3.0, -1.0], [2.0, -2.0, 5.0]])
rhs = system @ expected
solved = np.linalg.solve(system, rhs)
print("solved coefficients:", np.round(solved, 6))

solved coefficients: [ 1. -2.  3.]


## Fit a quadratic and factor its design matrix

Use least squares for the fit and QR for an independent reconstruction.

In [3]:
x = np.linspace(-2.0, 2.0, 9)
design = np.column_stack((np.ones_like(x), x, x * x))
observations = 2.0 - 3.0 * x + 0.5 * x * x
fitted, _, _, _ = np.linalg.lstsq(design, observations, rcond=None)
q, r = np.linalg.qr(design)
reconstruction = q @ r
print("quadratic fit:", np.round(fitted, 6))
print("QR diagonal:", np.round(np.diag(r), 6))

quadratic fit: [ 2.  -3.   0.5]
QR diagonal: [-3.        3.872983  4.387482]


## Extract PCA directions

Center a small dataset, eigendecompose its covariance, and compare with SVD.

In [4]:
samples = np.array([
    [2.0, 1.0, 0.0], [3.0, 2.0, 1.0], [4.0, 1.0, 2.0],
    [5.0, 3.0, 1.0], [6.0, 4.0, 3.0], [7.0, 3.0, 4.0],
])
centered = samples - samples.mean(axis=0)
covariance = centered.T @ centered / (samples.shape[0] - 1)
eigenvalues, eigenvectors = np.linalg.eig(covariance)
order = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[order].real
principal_axis = eigenvectors[:, order[0]].real
anchor = np.argmax(np.abs(principal_axis))
principal_axis *= np.where(principal_axis[anchor] < 0.0, -1.0, 1.0)
_, singular_values, _ = np.linalg.svd(centered, full_matrices=False)
print("PCA eigenvalues:", np.round(eigenvalues, 6))
print("first principal axis:", np.round(principal_axis, 6))
print("singular values:", np.round(singular_values, 6))

PCA eigenvalues: [6.312954 0.6941   0.126279]
first principal axis: [0.738456 0.39462  0.54677 ]
singular values: [5.618253 1.862928 0.794605]


## Verify the result

In [5]:
assert np.allclose(solved, [1.0, -2.0, 3.0], rtol=0.0, atol=1e-12)
assert np.allclose(fitted, [2.0, -3.0, 0.5], rtol=0.0, atol=1e-12)
assert np.allclose(reconstruction[:, 0], np.ones(9), rtol=0.0, atol=1e-12)
assert np.allclose(singular_values, [5.61825335, 1.86292763, 0.79460468], rtol=0.0, atol=1e-8)
print("PASS — all linear-algebra assertions passed.")

PASS — all linear-algebra assertions passed.
